# Independent Training and Evaluation of Deep Learning Models

This commented notebook trains a deep learning classifier on the selected BCE feature set, evaluates it on an internal hold-out split, and then tests the trained model on three independent evaluation datasets.

## What this notebook does
1. Imports the required Python, TensorFlow, and scikit-learn libraries.
2. Splits and standardizes the training data.
3. Loads the main training set plus three independent test sets.
4. Builds several deep learning model definitions (FCNN, RNN, GRU, LSTM, CNN).
5. Trains **CNN by default** using the selected hyperparameters.
6. Evaluates the trained model on:
   - internal validation/test split
   - internal independent test set
   - external test set I
   - external test set II
7. Saves the fitted scaler and CSV files containing evaluation metrics.

## Important note
The notebook is currently configured to **train and evaluate the CNN model only**. The other model builders are already defined in the model cell. Comments have been added below to show how to switch the pipeline to FCNN, RNN, GRU, or LSTM without changing the overall logic.


## 1. Import libraries and utilities

This cell imports the packages required for:
- data handling (`pandas`, `numpy`)
- plotting (`matplotlib`)
- deep learning (`tensorflow`, `keras`)
- tuning / callbacks (`keras_tuner`, early stopping, checkpointing)
- preprocessing and evaluation (`scikit-learn`)
- saving preprocessing objects (`joblib`)

### Import audit
- **No critical missing imports** were found for the current notebook logic.
- A few imports appear **unused in the current execution path**, for example:
  - `math`, `time`, `random`, `datetime`
  - `Sequential` (because the model builders use `tf.keras.Sequential`)
  - `Dense`, `GRU`, `Input`, `LSTM`, `Conv1D`, `MaxPool1D`, `BatchNormalization`, `Flatten`, `Dropout` (imported explicitly but not called directly, since `tf.keras.layers...` is used instead)
  - `Adam`, `RMSprop`, `SGD` (optimizers are referenced through `tf.keras.optimizers`)
  - `RandomSearch`, `Hyperband`, `kt`
  - `MinMaxScaler`
  - `cross_val_score`, `cross_val_predict`, `StratifiedKFold`
  - `roc_curve`, `auc`, `ConfusionMatrixDisplay`, `classification_report`, `precision_recall_curve`
  - `chi2`, `itertools`

These can be trimmed later for cleanliness, but they do not prevent execution.


In [ ]:
# Library imports and version check
import os, math, time, random, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
print(__doc__)

import tensorflow as tf
print(tf.__version__)

# from tensorflow.python.keras.models import Sequential, load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Input, LSTM, Conv1D, MaxPool1D, BatchNormalization, Flatten, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from keras_tuner.tuners import RandomSearch, Hyperband
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import regularizers
import keras_tuner as kt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler

from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report, f1_score, precision_recall_curve, average_precision_score

from sklearn.feature_selection import chi2
import itertools
# from mlxtend.plotting import plot_decision_regions
import joblib

### Split and scale method

## 2. Helper function: split and standardize the dataset
This function applies `StandardScaler` to the feature matrix and then creates a train/test split for downstream deep learning training. The fitted scaler is returned so the same transformation can be reused on the independent test sets.

In [ ]:
# Helper function for standardization + train/test split
# Get splitted data for deep learning models models
def getSplitDataSet(X, y, ratio=0.2):
    
    scaler = StandardScaler()
    scaler.fit(X)
    X = scaler.transform(X)

    #split data into training and test data. 
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ratio, random_state=245)

    return X_train, X_test, y_train, y_test, scaler

## 3. Load the prepared feature sets
Update these file paths if you are running the notebook from a different directory. The notebook expects CSV files where the **first column is the label** and the remaining columns are the feature values.

In [ ]:
# Load the selected feature set for training and independent evaluation
'''
you can change the path of the dataset to your local path, or use the provided path if you have the same directory structure as mine.
You can also change the feature set to other feature sets like F1 or F2
The url_paths will be converted accordingly (either form features_selected folder or features folder that have all features)
'''
url_dataset ="features_selected/PSTPP/train.csv" 
df_main = pd.read_csv(url_dataset, header=0)

url_testset ="features_selected/PSTPP/test.csv" 
df_test = pd.read_csv(url_testset, header=0)

url_ext1 = "features_selected/PSTPP/ext1.csv"
df_ext1 = pd.read_csv(url_ext1, header=0)

url_ext2 = "features_selected/PSTPP/ext2.csv"
df_ext2 = pd.read_csv(url_ext2, header=0)



## 4. Prepare labels, split the main dataset, save the scaler, and transform all independent test sets
This cell:
- extracts labels and features from the loaded CSVs
- encodes labels into integer form
- creates the internal training/test split
- saves the fitted `StandardScaler`
- applies the same scaler to the internal and external independent test sets

In [ ]:
# Prepare labels/features, split the main dataset, and standardize all evaluation sets
# Prepare the data
# X = df_main.drop(columns=["label"]).values
# y = df_main["label"].values

# Alternative way to prepare the data
y = df_main.iloc[:,:1].values
X = df_main.iloc[:,1:].values

encoder = LabelEncoder()
labels = y = encoder.fit_transform(y.ravel())

# Split the training set into further training and testing sets for deep learning models training
X_train, X_test, y_train, y_test, _scaler = getSplitDataSet(X, y, ratio=0.1)

joblib.dump(_scaler, f"_standard_scalsr.pkl")

# Prepare the independent test set (internal test set)
ind_labels = df_test.iloc[:,:1].values
ind_features = df_test.iloc[:,1:].values
# Ensure labels are integers
ind_labels = ind_labels.astype(int)
# Label encode the labels
ind_labels = LabelEncoder().fit_transform(ind_labels.ravel())
# Scale features
ind_features = _scaler.transform(ind_features)

# Prepare the External test set I
ext1_labels = df_ext1.iloc[:,:1].values
ext1_features = df_ext1.iloc[:,1:].values
# Ensure labels are integers
ext1_labels = ext1_labels.astype(int)
# Label encode the labels
ext1_labels = LabelEncoder().fit_transform(ext1_labels.ravel())
# Scale features
ext1_features = _scaler.transform(ext1_features)

# Prepare the External test set II
ext2_labels = df_ext2.iloc[:,:1].values
ext2_features = df_ext2.iloc[:,1:].values
# Ensure labels are integers
ext2_labels = ext2_labels.astype(int)
# Label encode the labels
ext2_labels = LabelEncoder().fit_transform(ext2_labels.ravel())
# Scale features
ext2_features = _scaler.transform(ext2_features)

fea_dim = X_train.shape[1]
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

## 5. Compute class weights
Class weights are used during training to reduce the effect of class imbalance. These weights are later passed to `model.fit(...)`.

In [ ]:
# Compute balanced class weights from the training labels
from sklearn.utils import class_weight
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = {0: cw[0], 1: cw[1]}
print(cw_dict)

## 7. Prepare reshaped inputs for different neural architectures
The same standardized feature matrix can be reshaped differently depending on the model type.

- **CNN** uses shape `(samples, features, 1)`
- **RNN/GRU/LSTM** use shape `(samples, 1, features)`
- **FCNN** uses the original 2D shape `(samples, features)`

These reshaped arrays are prepared here so you can easily switch architectures later.

In [ ]:
# Create architecture-specific input tensors
# Reshaping for CNN models
XTrainCNN =np.array(X_train).reshape(-1, fea_dim, 1)
XTestCNN = np.array(X_test).reshape(-1, fea_dim, 1)
print("XTrainCNN Shape",XTrainCNN.shape)

# Reshaping for RNN models
XTrainRNN =np.array(X_train).reshape(-1, 1, fea_dim)
XTestRNN = np.array(X_test).reshape(-1, 1, fea_dim)
print("XTrainRNN Shape",XTrainRNN.shape)

## 8. Define deep learning model builders
This cell defines all model architectures used in the study:
- FCNN
- RNN
- GRU
- LSTM
- CNN

### Current default
The later training cell uses **`build_cnn(...)`** by default.

### How to use another model
You do **not** need to rewrite the training logic. Only replace the model builder and input tensors in the training/evaluation cells. For example:

- **FCNN**
  - model: `build_fcnn(input_shape=X_train.shape[1])`
  - train input: `X_train`
  - test input: `X_test`
  - independent inputs: `ind_features`, `ext1_features`, `ext2_features`

- **RNN**
  - model: `build_rnn(input_shape=X_train.shape[1])`
  - train input: `XTrainRNN`
  - test input: `XTestRNN`
  - independent inputs must also be reshaped to `(-1, 1, fea_dim)`

- **GRU**
  - model: `build_gru(input_shape=X_train.shape[1])`
  - same reshaping strategy as RNN

- **LSTM**
  - model: `build_lstm(input_shape=X_train.shape[1])`
  - same reshaping strategy as RNN

A note has been inserted in the training cell showing exactly where to change this.

In [ ]:
# Model-definition cell: keep all builders here so the same downstream pipeline can be reused
# ========== MODEL BUILDERS ==========
def build_fcnn(input_shape):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(input_shape,)),
            tf.keras.layers.Dense(
                352,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(
                352,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", "AUC"],
    )
    return model

def build_rnn(input_shape):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Reshape((1, input_shape), input_shape=(input_shape,)),
            tf.keras.layers.RNN(
                tf.keras.layers.SimpleRNNCell(
                    units=48,
                    activation="tanh",
                    use_bias=True,
                    kernel_regularizer=tf.keras.regularizers.l2(0.0001),
                ),
            ),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(
                320,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(0.001),
            ),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01),
        loss="binary_crossentropy",
        metrics=["accuracy", "AUC"],
    )
    return model

def build_gru(input_shape):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Reshape((1, input_shape), input_shape=(input_shape,)),
            tf.keras.layers.GRU(
                32,
                activation="tanh",
                recurrent_activation="sigmoid",
                use_bias=True,
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
                reset_after=False, # <- for better performance on small datasets
                recurrent_dropout=0.1,  # <- also disables CuDNN
                implementation=2,   # <- standard kernel
                return_sequences=False,
            ),
            tf.keras.layers.Dropout(0.1),
            tf.keras.layers.Dense(
                32,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", "AUC"],
    )
    return model

def build_lstm(input_shape):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Reshape((1, input_shape), input_shape=(input_shape,)),
            tf.keras.layers.LSTM(
                96,
                activation="tanh",
                recurrent_activation="sigmoid",
                kernel_regularizer=tf.keras.regularizers.l2(0.0001),
                use_bias=True,
                recurrent_dropout=0.1, # <- disables CuDNN
                implementation=2,  # <- standard kernel
                return_sequences=True,
            ),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.LSTM(
                96,
                activation="tanh",
                recurrent_activation="sigmoid",
                kernel_regularizer=tf.keras.regularizers.l2(0.0001),
                use_bias=True,
                recurrent_dropout=0.1,  # <- disables CuDNN
                implementation=2,  # <- standard kernel
                return_sequences=False,
            ),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(
                32,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", "AUC"],
    )
    return model

def build_cnn(input_shape):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Reshape((input_shape, 1), input_shape=(input_shape,)),
            tf.keras.layers.Conv1D(
                filters=64,
                kernel_size=8,
                activation="relu",
                padding="same",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.AveragePooling1D(),#MaxPool1D(2),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.4),
            tf.keras.layers.Conv1D(
                filters=32,
                kernel_size=8,
                activation="relu",
                padding="same",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.AveragePooling1D(),#MaxPooling1D(2),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(
                192,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.4),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0005),
        loss="binary_crossentropy",
        metrics=["accuracy", "AUC"],
    )
    return model

## 9. Training callbacks
These callbacks:
- save the best-performing model checkpoint
- reduce the learning rate when validation accuracy plateaus
- restore the best weights if training stops early

In [ ]:
# Callback configuration for model training
file_path = "saved_models/best_cnn_model.h5"
os.makedirs(os.path.dirname(file_path), exist_ok=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
                file_path,
                monitor="val_accuracy",
                verbose=1,
                save_best_only=True,
                mode="max",
                save_weights_only=False,
            )
reduce_on_plateau = tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_accuracy",
                mode="max",
                factor=0.5,
                patience=10,
                verbose=1,
                min_lr=0.00002,
            )
restore_best_weights = tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                patience=50,
                restore_best_weights=True,
                verbose=1,
                mode="max",
            )
callbacks_list = [checkpoint, reduce_on_plateau, restore_best_weights]

## 10. Train the model on the internal training split and evaluate on the internal hold-out test split
This cell trains the **CNN model by default** and saves the internal test metrics.

### To switch to another architecture
Use the same logic, but replace:
- the model builder (`build_cnn(...)`) with the desired builder
- the input arrays with the correct shapes

Examples:
- FCNN: use `X_train`, `X_test`
- CNN: use `X_train`, `X_test` directly here because the CNN builder internally reshapes
- RNN/GRU/LSTM: either modify the builder to accept already reshaped tensors or feed appropriately reshaped arrays consistently throughout training and prediction

If you prefer not to duplicate cells, you can simply rerun this cell after changing only the model-construction line and the train/test tensors used.

In [ ]:
# Train on the internal split and evaluate on the hold-out test subset
results = []  ### rename to results_train

# Convert data/features to tf.data.Dataset format for better performance
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Build the model
# Default choice: CNN
# To switch architectures, replace the line below with one of:
# model = build_fcnn(input_shape=X_train.shape[1])
# model = build_rnn(input_shape=X_train.shape[1])
# model = build_gru(input_shape=X_train.shape[1])
# model = build_lstm(input_shape=X_train.shape[1])
model = build_cnn(input_shape=X_train.shape[1])
# Train the model
model.fit(
                train_dataset.batch(16),
                epochs=500,
                batch_size=16,
                callbacks=callbacks_list,
                verbose=1,
                validation_data=test_dataset.batch(16),
                class_weight=cw_dict,
            )

# Predict probabilities and classes
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

# Metrics for test set
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
_auc = roc_auc_score(y_test, y_pred_prob)
mAP = average_precision_score(y_test, y_pred_prob)
mcc = matthews_corrcoef(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

# Calculate Sensitivity
sens = TP / (TP + FN)

# Calculate Specificity
spec = TN / (TN + FP)

# Save results
results.append(
                {
                    "Accuracy": acc,
                    "F1_Score": f1,
                    "Sensitivity": sens,
                    "Specificity": spec,
                    "mAP": mAP,
                    "AuROC": _auc,
                    "MCC": mcc,
                }
            )
results_df = pd.DataFrame(results)
results_csv_path = os.path.join("results", f"internal_results.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to {results_csv_path}")

## 11. Save the final trained model
This cell saves the trained Keras model so it can be reloaded later for inference or independent testing without redefining the architecture.

In [ ]:
# Save the trained model for later inference / deployment
path="saved_models/"
model_save_path = f"{path}/final_cnn_model.h5"
os.makedirs(path, exist_ok=True)
model.save(model_save_path)
print(f"Trained model saved to {model_save_path}")

## 12. Evaluate on the internal independent test set
This dataset is separate from the training split and is used as the first independent evaluation benchmark.

In [ ]:
# Evaluate the trained model on the internal independent test set
'''
Internal test set is used as the independent test set for evaluation, and the two external test sets will be used as the independent test sets for further evaluation in the next steps.
'''

# Predict
y_pred_prob = model.predict(ind_features).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)
# Metrics
f1 = f1_score(ind_labels, y_pred)
acc = accuracy_score(ind_labels, y_pred)
_auc = roc_auc_score(ind_labels, y_pred_prob)
mAP = average_precision_score(ind_labels, y_pred_prob)
mcc = matthews_corrcoef(ind_labels, y_pred)
cm = confusion_matrix(ind_labels, y_pred)
TN, FP, FN, TP = cm.ravel()
# Calculate Sensitivity
sens = TP / (TP + FN)
# Calculate Specificity
spec = TN / (TN + FP)
# Save results
ind_results = []
ind_results = {
                    "Accuracy": acc,
                    "F1_Score": f1,
                    "Sensitivity": sens,
                    "Specificity": spec,
                    "mAP": mAP,
                    "AuROC": _auc,
                    "MCC": mcc,
                }
ind_results_df = pd.DataFrame([ind_results])
results_csv_path = os.path.join("results", f"ind_results.csv")
ind_results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to {results_csv_path}")

## 13. Evaluate on External Test Set I
This cell reuses the same trained model and scaler to assess generalization on the first external benchmark.

In [ ]:
# Evaluate the trained model on External Test Set I
'''
External test set I is used as the independent test set for evaluation, and the two external test sets will be used as the independent test sets for further evaluation in the next steps.
'''
# Predict
y_pred_prob = model.predict(ext1_features).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)
# Metrics
f1 = f1_score(ext1_labels, y_pred)
acc = accuracy_score(ext1_labels, y_pred)
_auc = roc_auc_score(ext1_labels, y_pred_prob)
mAP = average_precision_score(ext1_labels, y_pred_prob)
mcc = matthews_corrcoef(ext1_labels, y_pred)
cm = confusion_matrix(ext1_labels, y_pred)
TN, FP, FN, TP = cm.ravel()
# Calculate Sensitivity
sens = TP / (TP + FN)
# Calculate Specificity
spec = TN / (TN + FP)
# Save results
ext1_results = []
ext1_results = {
                    "Accuracy": acc,
                    "F1_Score": f1,
                    "Sensitivity": sens,
                    "Specificity": spec,
                    "mAP": mAP,
                    "AuROC": _auc,
                    "MCC": mcc,
                }
ext1_results_df = pd.DataFrame([ext1_results])
results_csv_path = os.path.join("results", f"ext1_results.csv")
ext1_results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to {results_csv_path}")


## 14. Evaluate on External Test Set II
This cell performs the final independent evaluation on the second external benchmark.

In [ ]:
# Evaluate the trained model on External Test Set II
'''
External test set II is used as the independent test set for evaluation, and the two external test sets will be used as the independent test sets for further evaluation in the next steps.
'''
# Predict
y_pred_prob = model.predict(ext2_features).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)
# Metrics
f1 = f1_score(ext2_labels, y_pred)
acc = accuracy_score(ext2_labels, y_pred)
_auc = roc_auc_score(ext2_labels, y_pred_prob)
mAP = average_precision_score(ext2_labels, y_pred_prob)
mcc = matthews_corrcoef(ext2_labels, y_pred)
cm = confusion_matrix(ext2_labels, y_pred)
TN, FP, FN, TP = cm.ravel()
# Calculate Sensitivity
sens = TP / (TP + FN)
# Calculate Specificity
spec = TN / (TN + FP)
# Save results
ext2_results = []
ext2_results = {
                    "Accuracy": acc,
                    "F1_Score": f1,
                    "Sensitivity": sens,
                    "Specificity": spec,
                    "mAP": mAP,
                    "AuROC": _auc,
                    "MCC": mcc,
                }
ext2_results_df = pd.DataFrame([ext2_results])
results_csv_path = os.path.join("results", f"ext2_results.csv")
ext2_results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to {results_csv_path}")
